# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivors dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library. We'll follow the Croissant schema to review record sets, fields, and their IDs, and load data for basic exploratory analysis.

### Dataset Source
This dataset's Croissant schema is hosted at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and their `@id` values, along with the fields (columns) in each, as defined in the Croissant schema.

In [ ]:
# List all record sets and their fields with @id values

croissant_schema = dataset.schema

print("Available record sets in the dataset:")

record_sets = []
for obj in croissant_schema.get('recordSet', []):
    rec_id = obj.get('@id')
    record_sets.append(rec_id)
    name = obj.get('name', '[no name]')
    print(f"  RecordSet @id: {rec_id}  (name: {name})")
    fields = obj.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for field in fields:
        field_id = field.get('@id')
        label = field.get('name', '[no name]')
        print(f"      Field @id: {field_id}  (name: {label})")
    print()
if not record_sets:
    print("  No record sets found in schema.")

### For demonstration, let's view the first few records (rows) for each record set using their `@id` values.

_Note: Replace `<record_set_id>` below with the relevant record set `@id` printed above._

In [ ]:
# Print a few sample records from each record set using its @id
for record_set_id in record_sets:
    print(f"\nFirst 2 records for RecordSet @id: {record_set_id}")
    try:
        for k, x in enumerate(dataset.records(record_set=record_set_id)):
            print(x)
            if k >= 1:
                break
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")

## 3. Data Extraction
Next, we'll extract all record sets into Pandas DataFrames using their exact `@id` values.

In [ ]:
# Extract each available record set into a DataFrame
dataframes = {}

for record_set_id in record_sets:
    print(f"\nLoading {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print("  No records available for this record set.")
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")

# If there is at least one record set with records, choose it for later analysis
for candidate_id in dataframes:
    main_record_set_id = candidate_id
    main_df = dataframes[main_record_set_id]
    break
else:
    main_record_set_id = None

if main_record_set_id:
    print(f"\nWill use main RecordSet: {main_record_set_id}")
    print(main_df.columns.tolist())
    display(main_df.head())
else:
    print("No available data for analysis.")

## 4. Exploratory Data Analysis (EDA)
Let's filter and transform data for analysis. We'll select a numeric field and a grouping field by their `@id`. If numeric and groupable fields are present, we will filter by a threshold, normalize, and group.

In [ ]:
# -- Parameters: set IDs of numeric field and group field for analysis --
# You should review available fields above and replace the values accordingly.

# Example: If a numeric field is '@id': 'num_years_between_diagnoses'
numeric_field = None  # Example: 'cr:num_years_between_diagnoses'
group_field = None    # Example: 'cr:sex'
main_df = None

if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    # Attempt to auto-detect a numeric field
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break
    # Attempt to auto-detect a groupable (categorical) field
    for col in main_df.columns:
        if main_df[col].dtype == object and main_df[col].nunique() > 1:
            group_field = col
            break
    
    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        threshold = main_df[numeric_field].mean() if main_df[numeric_field].dtype in [int,float] else 0
        # Filter
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_name = f"{numeric_field}_normalized"
        filtered_df[norm_name] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_name]].head())

        # Group by a discrete (categorical) field, if available
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field available for grouping.")
    else:
        print("No numeric field found for EDA. Please check the dataset fields above and set 'numeric_field' manually.")
else:
    print("No main data available for EDA.")

## 5. Visualization
Visualize distribution of the selected numeric field and relationship with group field, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_field], showmeans=True)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded a FAIR² colorectal cancer survivors dataset using the `mlcroissant` library, explored its record sets and field identifiers, and extracted data into DataFrames. We applied basic EDA, including filtering and group analysis on available numeric and categorical fields, and visualized the distributions. 

To perform deeper analysis, review the schema output and select specific `@id` field(s) of clinical interest for your subsequent workflows.